# Inter-USV interval analyses

This notebook fits and visualises mixture models on the distribution of **inter-USV intervals** (in seconds, log-transformed) for one or more sessions. It produces:

1. A **master DataFrame** of every interval across the chosen session list.
2. An **information-criterion sweep** over candidate component counts for the selected mixture family (Gaussian, Student-t or inverse-Gaussian, per `model_class`).
3. A **bootstrap likelihood-ratio test** that selects the best `K` per family.
4. The **fit plots**: log-interval histogram with overlay, Q-Q diagnostic, and a table of fitted component parameters per sex.

Compute is split from plotting so you can re-render figures without refitting.

In [ ]:
### Imports and color settings

# Auto-reload package modules whenever they change on disk, so edits to
# the usv_playpen package are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pls

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style
from usv_playpen.visualizations.figure_io import save_figure
from usv_playpen.analyses.compute_inter_usv_interval_distributions import InterUSVIntervalCalculator
import usv_playpen.visualizations.usv_interval_summary_statistics as ivs

apply_plot_style()

base_path = Path.cwd().parent
with open(base_path / "_parameter_settings" / "visualizations_settings.json", 'r') as vis_settings_file:
    vis_settings = json.load(vis_settings_file)

with open(base_path / "_parameter_settings" / "analyses_settings.json", 'r') as ana_settings_file:
    ana_settings = json.load(ana_settings_file)

male_color = vis_settings["male_colors"][0]
female_color = vis_settings["female_colors"][0]

# All compute / display knobs live in analyses_settings.json -> compute_inter_usv_interval_distributions
usv_interval_cfg = ana_settings["compute_inter_usv_interval_distributions"]

## Configuration

Set the session list(s) to include, the candidate component counts, and the bootstrap size. The bootstrap step is the only slow part of the notebook (cell **[5]**).


In [ ]:
### Configuration

# Single source of truth for paths and labels. All numeric parameters live
# in analyses_settings.json; this cell only assigns notebook-local convenience
# aliases.
output_directory = usv_interval_cfg["output_directory"]
session_lists = [str(Path(configure_path(p))) for p in usv_interval_cfg["session_lists"]]

interval_types = ("s2s", "e2s")
mode_label = {
    "s2s": "start-to-start USV intervals",
    "e2s": "end-to-start USV intervals",
}

# Plot-only knobs (read straight from JSON; plot_log_xlims/bins_per_sex not archived in the HDF5, tau is)
plot_log_xlims = tuple(usv_interval_cfg["plot_log_xlims"])
bins_per_sex = usv_interval_cfg["bins_per_sex"]
tau = usv_interval_cfg["tau"]
model_class = usv_interval_cfg["model_class"]

print(f"output_directory: {output_directory}")
print(f"session_lists ({len(session_lists)}):")
for p in session_lists:
    print(f"  {p}")
print(f"model_class: {model_class!r}")

# Interval definitions are both COMPUTED, but only e2s is drawn unless asked for:
# the tied peaks-plus-background model is defined on end-to-start gaps, and
# start-to-start intervals carry each call's own duration, which smears the gap
# structure (the s2s peak test runs to the top of its grid with broad "peaks").
# Add "s2s" here to draw it anyway.
plot_interval_types = ("e2s",)
interval_label = {
    "s2s": "start-to-start intervals",
    "e2s": "end-to-start intervals",
}
call_label = {"usv": "USV", "squeak": "squeak"}

## Compute

Run once. The cell below runs `InterUSVIntervalCalculator` -- the same code the `generate-usv-interval-distributions` CLI runs -- with the settings in `analyses_settings.json`, so the notebook and the CLI write identical archives. It writes one self-describing HDF5 archive, `usv_interval_analysis_<YYYYMMDD>_<HHMMSS>.h5`, to `output_directory`; after that the plot cells read the most-recent archive on disk, so plot styling can be iterated across kernel restarts. The run takes about an hour, dominated by the session-corrected peak-count test.

What it does, per interval type (`s2s`, `e2s`) and per pool declared in `interval_pools`:

- builds the pool -- the male's ultrasonic calls, the female's squeaks -- with the pool's call type and adjacency rule, noise excluded;
- for pools marked `fit` and above `min_intervals_for_fitting`: fits the tied-scale Student-t ladder (peaks sharing one width plus `tied_n_background` free background components) over `tied_peak_grid`, and runs the step-up peak-count bootstrap LRT with each rung's statistic divided by its session design effect;
- for every pool: writes descriptive statistics.

The archive layout (see `usv_playpen.analyses.usv_interval_archive`):

- `/attrs` -- every JSON parameter that drove the run, plus `created_at_iso`, `git_sha`, `source_lists`, `n_sessions_loaded`.
- `/<mode>/intervals` -- tidy one-row-per-interval table, with `sex`, `call_type` and `adjacency` identifying the pool.
- `/<mode>/pool_summary` -- per-pool counts and quantiles, and whether the pool was fitted.
- `/<mode>/tied_fits`, `/<mode>/tied_modes` -- the tied ladder, one row per (peak count, component) and per density mode.
- `/<mode>/peak_lrt`, `/<mode>/peak_lrt_null` -- per-rung raw and corrected statistics, design effects, p-values, and the null draws.
- `/<mode>/attrs` -- the selected peak count and the per-rung significance level for each fitted pool.

The unconstrained component sweep (`mixture_model_fits`, `bootstrap_lrt`) is written only when `fit_mixture_model` is true.

In [ ]:
### [Compute] Run the interval analysis and write the archive

InterUSVIntervalCalculator(
    input_parameter_dict=ana_settings,
    message_output=print,
).save_inter_usv_interval_distributions_to_file()

h5_path = ivs.find_latest_archive(output_directory)
print(f"archive written: {h5_path}")

## Plot

These cells **read everything from the HDF5 archive** in `output_directory`,
so you can iterate on plot styling without re-running anything in the Compute
section -- even across kernel restarts. By default we pick the most-recent
archive (`ivs.find_latest_archive`); set `h5_path` manually to re-render an
older run.

## Plot step 1 — Output controls

Choose the archive to plot from and where to write figures. Switching archives lets you produce comparison figures across cohorts without re-fitting.


In [ ]:
### Figure-saving controls + select archive to read from

save_fig_bool = False

# Pick up the most-recent archive so plot cells survive kernel restarts.
# Override by assigning `h5_path = Path("...absolute/path/usv_interval_analysis_xxx.h5")`.
h5_path = ivs.find_latest_archive(output_directory)
print(f"reading from: {h5_path}")

## Plot step 2 — log-interval histogram (sanity check)

First diagnostic: a histogram of `log(interval)` per sex with the empirical kernel density. Confirms that the data is bimodal-ish (short intra-bout intervals + long inter-bout intervals) before any model is fit.


In [ ]:
### [Plot] log_interval distribution (sanity check)

# One axis per interval type, both pools overlaid. The pools are different
# vocalizations -- the male's ultrasonic calls and the female's squeaks -- so the
# legend names the call type rather than the sex alone.
for it in plot_interval_types:
    sub = ivs.load_intervals_from_h5(str(h5_path), it)
    pools = {row["sex"]: row["call_type"]
             for row in ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"].iter_rows(named=True)}
    fig, ax, hist_stats = ivs.plot_log_usv_interval_histograms(
        usv_interval_df=sub,
        bins=max(bins_per_sex.values()),
        male_color=male_color,
        female_color=female_color,
        xlims=plot_log_xlims,
        male_label=f"male {call_label[pools['male']]}",
        female_label=f"female {call_label[pools['female']]}",
    )
    ax.set_title(f"log_interval distribution -- {interval_label[it]}")
    if save_fig_bool:
        save_figure(fig, f"ivi_log_histogram_{it}", vis_settings)
    plt.show()
    display({**hist_stats, "interval_type": it})

## Plot step 3 — Bootstrap LRT null distribution

The session-corrected peak-count test, one panel per rung (1 vs 2 peaks, 2 vs 3, 3 vs 4) on each fitted pool. The grey histogram is the parametric bootstrap null; the red line is the observed likelihood ratio **divided by its session design effect** (intervals are nested in sessions and the null is not), and the dotted line is the null quantile at the Bonferroni-corrected level. The step-up rule stops at the first rung that keeps.

In [ ]:
### [Plot] Bootstrap LRT null-distribution panel (session-corrected peak-count test)

# One panel per rung of the peak-count test on each fitted pool. The red line is
# the CORRECTED statistic -- the observed likelihood ratio divided by its session
# design effect -- and the dotted line is the null quantile at the
# Bonferroni-corrected level the test used; the p-value is the corrected one.
for it in plot_interval_types:
    summary = ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"]
    for row in summary.filter(pls.col("fitted")).iter_rows(named=True):
        sweep, alpha_used = ivs.load_peak_lrt_sweep_from_h5(str(h5_path), it, row["sex"], row["call_type"])
        fig, _ = ivs.plot_bootstrap_lrt_panel(
            sweep,
            alpha=alpha_used,
            pair_title="{key}: {K_n} vs {K_a} peaks",
            obs_label="LR corrected",
            threshold_label="corrected 99%",
        )
        fig.suptitle(f"Bootstrap LRT null distributions -- {row['sex']} "
                     f"{call_label[row['call_type']]} {interval_label[it]}", y=1.08)
        fig.tight_layout()
        if save_fig_bool:
            save_figure(fig, f"ivi_bootstrap_lrt_{it}", vis_settings)
        plt.show()
        _, _, n_peak, _ = ivs.load_tied_model_from_h5(str(h5_path), it, row["sex"], row["call_type"])
        print(f"[{interval_label[it]}] step-up (session-corrected) selected: {n_peak} peak(s)")

## Plot step 4 — BIC vs number of peaks

BIC of each fit on the tied ladder (background count fixed), with the peak count selected by the corrected test outlined. BIC keeps falling past the selected count because it charges each parameter `log(n)` at the nominal `n`, which over-counts the evidence when intervals are nested in sessions.

In [ ]:
### [Plot] BIC vs peak count

# BIC of each fit on the tied ladder (background count fixed), with the peak count
# the corrected step-up test selected outlined. BIC keeps falling past the
# selected count: it charges each parameter log(n) at the nominal n, which
# over-counts the evidence when intervals are nested in sessions -- the same
# clustering the corrected test accounts for.
for it in plot_interval_types:
    summary = ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"]
    for row in summary.filter(pls.col("fitted")).iter_rows(named=True):
        df_ic = ivs.load_tied_ic_table_from_h5(str(h5_path), it, row["sex"], row["call_type"])
        _, _, n_peak, _ = ivs.load_tied_model_from_h5(str(h5_path), it, row["sex"], row["call_type"])
        fig, (ax_left, ax_right), _ = ivs.plot_ic_curves(
            df_results=df_ic,
            male_color=male_color,
            female_color=female_color,
            ic_col="bic",
            selected_n_components={row["sex"]: n_peak},
        )
        ax_left.set_xlabel("number of peaks")
        ax_left.set_title(f"BIC vs number of peaks -- {row['sex']} "
                          f"{call_label[row['call_type']]} {interval_label[it]}")
        if save_fig_bool:
            save_figure(fig, f"ivi_bic_curve_{it}", vis_settings)
        plt.show()

## Plot step 5 — Best-fit overlay

Histogram of `log(interval)` with the selected tied-scale mixture (peaks sharing one width, plus free background components) overlaid; an inset Q-Q plot compares empirical to model quantiles. `show_mixture_components` switches the individual weighted components on or off.

In [ ]:
### [Plot] Best-fit overlay (with Q-Q inset) at the test-selected peak count

# The tied-scale model rebuilt from the archive at the peak count the corrected
# step-up test selected -- no refit. Triangles mark each component, (a)/(b)/...
# ascending, with medians in the text legend. Only fitted pools are drawn.
#
# Toggle: when True, overlay each individual mixture component (its weighted
# density, so the components sum to the black mixture curve) on the histogram;
# when False only the mixture curve is drawn.
show_mixture_components = False
color_for = {"male": male_color, "female": female_color}
model_class_label = {"t": "t-distribution", "gauss": "Gaussian", "ig": "inverse-Gaussian"}.get(model_class, model_class)
density_as_bin_means = usv_interval_cfg["density_as_bin_means"]

for it in plot_interval_types:
    sub = ivs.load_intervals_from_h5(str(h5_path), it)
    summary = ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"]
    for row in summary.filter(pls.col("fitted")).iter_rows(named=True):
        sex, call_type = row["sex"], row["call_type"]
        intervals_sec = sub.filter((pls.col("sex") == sex) & (pls.col("call_type") == call_type))["interval_s"].to_numpy()
        mixture_model, mixture_model_order, n_peak, rows = ivs.load_tied_model_from_h5(
            str(h5_path), it, sex, call_type,
        )
        n_background = int(rows["n_background"][0])
        kw = (dict(auto_inset_below_legend=True, auto_inset_size=(0.45, 0.45)) if sex == "male"
              else dict(auto_inset_below_legend=True, auto_inset_size=(0.25, 0.25)))
        fig_fit, ax_fit, fit_summary = ivs.plot_best_fit_with_annotations(
            intervals_sec=intervals_sec,
            mixture_model=mixture_model,
            mixture_model_order=mixture_model_order,
            color=color_for[sex],
            figsize=(5, 5),
            bins=bins_per_sex[sex],
            xlims=plot_log_xlims,
            tau=tau,
            legend_corner="upper right",
            show_components=show_mixture_components,
            density_as_bin_means=density_as_bin_means,
            **kw,
        )
        ax_fit.set_title(
            f"{interval_label[it].capitalize()} of {sex} {call_label[call_type]}s -- {model_class_label} "
            f"mixture model LRT-selected {n_peak} peaks + {n_background} background"
        )
        if save_fig_bool:
            save_figure(fig_fit, f"ivi_best_fit_{model_class}_{sex}_{it}", vis_settings)
        plt.show()
        print(f"  [{sex}] log-log Pearson r (Q-Q) = {fit_summary['qq_pearson_r']:.4f}")

## Plot step 6 — Print fitted parameters

Log-means and log-scales of the selected tied model's components, ascending in location, in the `mixture_model_params` layout the modeling pipeline reads (component 0 is the first peak, from which the inter-bout threshold `exp(mu_0 + 2.58 * sd_0)` is derived).

In [ ]:
### [Print] Fitted component parameters (log-mean / log-scale, ascending)

import json

for it in plot_interval_types:
    summary = ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"]
    blocks = []
    for row in summary.filter(pls.col("fitted")).iter_rows(named=True):
        _, _, n_peak, rows = ivs.load_tied_model_from_h5(str(h5_path), it, row["sex"], row["call_type"])
        rows = rows.sort("logmean")
        means = [round(float(m), 5) for m in rows["logmean"]]
        sds = [round(float(s), 5) for s in rows["logscale"]]
        roles = rows["role"].to_list()
        blocks.append(
            f'    "{row["sex"]}": {{\n'
            f'      "means": {json.dumps(means)},\n'
            f'      "sds": {json.dumps(sds)}\n'
            f'    }}'
        )
        threshold_ms = 1000.0 * float(np.exp(means[0] + 2.58 * sds[0]))
        print(f"[{it}] {row['sex']} {call_label[row['call_type']]}: roles {roles}; "
              f"inter-bout threshold exp(mu_0 + 2.58 sd_0) = {threshold_ms:.1f} ms")
    print("{")
    print('  "mixture_model_params": {')
    print(",\n".join(blocks))
    print("  }")
    print("}")

## Plot step 7 — Serial dependence

The mixture says where the components are; it does not say which of them marks the boundary between a within-bout pause and a between-bout gap. That is settled by **serial dependence**, the defining property of a bout: a gap inside a bout carries information about the gap that follows it, a gap between bouts does not.

The panel is the joint distribution of consecutive intervals (current on x, next on y, same log scale, square box), with the median next interval in equal-count bins of the current one. Within a bout the median runs along the `next = current` diagonal; once a gap exceeds the bout boundary -- the first peak's `exp(mu + 2.58 sd)`, the same bound the modeling pipeline uses -- the median goes flat, because the length of a long gap no longer predicts the typical length of the next. The coupling fades gradually rather than switching off, so the boundary marks where it is gone, not a sharp transition.

In [ ]:
### [Plot] Serial dependence between consecutive intervals

# Pairs are consecutive rows of the archived interval table within a session, which
# is the order the analysis wrote them (ascending in time). The boundary is the first
# peak's exp(mu + z * sd) with the same z the modeling pipeline uses for its
# inter-bout threshold, so the line drawn here IS that threshold.
color_for = {"male": male_color, "female": female_color}
serial_dependence_z = 2.58

for it in plot_interval_types:
    sub = ivs.load_intervals_from_h5(str(h5_path), it)
    summary = ivs.read_usv_interval_h5(str(h5_path))["modes"][it]["pool_summary"]
    for row in summary.filter(pls.col("fitted")).iter_rows(named=True):
        sex, call_type = row["sex"], row["call_type"]
        pool = sub.filter((pls.col("sex") == sex) & (pls.col("call_type") == call_type))
        pairs = ivs.serial_dependence_pairs(pool, sex)
        _, _, _, rows = ivs.load_tied_model_from_h5(str(h5_path), it, sex, call_type)
        first_peak = rows.filter(pls.col("role") == "peak").sort("logmean").row(0, named=True)
        boundary_ms = 1000.0 * float(np.exp(first_peak["logmean"] + serial_dependence_z * first_peak["logscale"]))

        fig_sd, ax_sd, sd_stats = ivs.plot_serial_dependence(
            pairs=pairs,
            color=color_for[sex],
            boundary_ms=boundary_ms,
        )
        if save_fig_bool:
            save_figure(fig_sd, f"ivi_serial_dependence_{sex}_{it}", vis_settings)
        plt.show()

        beyond = sd_stats["bin_median_next_ms"][sd_stats["bin_current_ms"] > boundary_ms]
        print(f"  [{it}] [{sex} {call_label[call_type]}]: {sd_stats['n_pairs']} pairs; "
              f"bout boundary {boundary_ms:.0f} ms; beyond it the median next interval stays within "
              f"{beyond.min():.0f}-{beyond.max():.0f} ms" if beyond.size else "")